In [ ]:
sns.set_theme(style="ticks")

adata_ref = adata_ref_ref.copy()

genes = ['EGFP_Seq1', 'tdTomato_Seq2']

def to_dense(x):
    return x.toarray().ravel() if hasattr(x, "toarray") else np.ravel(x)

# Raw counts for filtering
egfp_raw_all = to_dense(adata_ref[:, "EGFP_Seq1"].X)
td_raw_all   = to_dense(adata_ref[:, "tdTomato_Seq2"].X)

# Filter: only cells with >1 raw count in BOTH genes
bg_filter = (egfp_raw_all > 1) & (td_raw_all > 1)

gauss_results = {}

# -------- Fit a single Gaussian to log1p(counts) --------
for g in genes:
    x_raw_full = to_dense(adata_ref[:, g].X)
    x_raw = x_raw_full[bg_filter]

    # log1p transform
    x_log = np.log1p(x_raw)

    # Fit a simple one-component Gaussian
    mu = np.mean(x_log)
    sigma = np.std(x_log)

    # 99th percentile of Gaussian = mean + 2.326 * sd
    thr_log = mu + 2.326 * sigma

    # call positives
    hi_mask = np.zeros(adata_ref.n_obs, dtype=bool)
    hi_mask[bg_filter] = x_log > thr_log
    adata_ref.obs[f"{g}_hi"] = hi_mask

    gauss_results[g] = {
        "x_log": x_log,
        "thr_log": thr_log,
        "mu": mu,
        "sigma": sigma
    }

# Dual-high
adata_ref.obs['EGFP_tdTomato_dual_hi'] = (
    adata_ref.obs['EGFP_Seq1_hi'] &
    adata_ref.obs['tdTomato_Seq2_hi']
)

dual = adata_ref.obs['EGFP_tdTomato_dual_hi'][bg_filter].values

# Extract for plotting
x_egfp_log = gauss_results["EGFP_Seq1"]["x_log"]
x_td_log   = gauss_results["tdTomato_Seq2"]["x_log"]
thr_egfp_log = gauss_results["EGFP_Seq1"]["thr_log"]
thr_td_log   = gauss_results["tdTomato_Seq2"]["thr_log"]

# -------- Plot --------
plt.figure(figsize=(5,5), dpi=150)
ax = plt.gca()

ax.scatter(x_egfp_log, x_td_log, s=8, color="gray", alpha=0.25)
ax.scatter(x_egfp_log[dual], x_td_log[dual],
           s=60, facecolors=(1,0,0,0.4), edgecolors="black")

ax.axvline(thr_egfp_log, color="#e41a1c", ls="--", lw=2)
ax.axhline(thr_td_log,   color="#e41a1c", ls="--", lw=2)

ax.set_xlabel("EGFP (log1p counts)")
ax.set_ylabel("tdTomato (log1p counts)")
ax.set_title("log1p counts — Gaussian threshold (>1 raw count subset)")

sns.despine()
plt.tight_layout()
plt.show()